# Interactive session with Spark to explore Bronze results

This notebook uses the project virtual environment and repo-local paths only.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "bronze").exists():
    repo_root = Path("/home/dcamacho/dev/ProjectData").resolve()

source_sample_dir = repo_root / "sample_data"
source_dir = repo_root / "data/exports/projectA"
table_path = repo_root / "_tmp" / "bronze_pid_documents"

# Remove stale Spark/Python overrides from previous runs
for key in ["PYTHONPATH", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    os.environ.pop(key, None)
    
sys.path = [p for p in sys.path if "/opt/spark" not in p]

print("repo_root =", repo_root)
print("source_sample_dir =", source_sample_dir)
print("source_dir =", source_dir)
print("table_path =", table_path)
print("python_executable =", sys.executable)

print("Python:", sys.executable)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

repo_root = /home/dcamacho/dev/ProjectData
source_sample_dir = /home/dcamacho/dev/ProjectData/sample_data
source_dir = /home/dcamacho/dev/ProjectData/data/exports/projectA
table_path = /home/dcamacho/dev/ProjectData/tmp/bronze_pid_documents
python_executable = /home/dcamacho/dev/ProjectData/.venv/bin/python
Python: /home/dcamacho/dev/ProjectData/.venv/bin/python
PYTHONPATH: None


In [2]:
import pyspark
print(pyspark.__file__)
print(pyspark.__version__)

/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py
3.5.1


In [3]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_sample_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code Sample

26/08/29 14:23:45 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/29 14:23:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-febd1df3-c97e-4191-9698-b6e45b5e4928;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 180ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-sto

In [4]:
# Keep the runtime clean for Spark. Do not set PYTHONPATH or SPARK_HOME manually here.
for key in ["PYTHONPATH", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    os.environ.pop(key, None)

print("Using Python:", sys.executable)
print("Python version:", sys.version)

Using Python: /home/dcamacho/dev/ProjectData/.venv/bin/python
Python version: 3.9.23 (main, Jun  4 2025, 08:55:38) 
[GCC 9.4.0]


In [5]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


In [6]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Interactive_Bronze_Exploration")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark ready:", spark.version)

your 131072x1 screen size is bogus. expect trouble


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0d4099bd-5691-4c75-ba9a-5ca7bf02926b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 193ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

Spark ready: 3.5.1


In [7]:
# Load the table created by the CLI ingest run
path = str(table_path)
df = spark.read.format("delta").load(path)
print("Rows:", df.count())
df.show(5, truncate=False)

26/08/29 14:24:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows: 4
+------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
# Show a compact preview
visible_columns = [
    "document_number",
    "drawing_revision",
    "project_code",
    "source_filename",
    "file_size_bytes",
    "ingested_at",
    "content_text",
]

preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

,document_number,drawing_revision,project_code,source_filename,file_size_bytes,ingested_at,content_text
0,215777C-36209-PID-0021-01010,01,Sample,projectA_dexpi_01010.xml,1040,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Synthetic DEXPI/Proteus P&ID export (project A). Structure is illustrative;\n confirm real elemen..."
1,A14-0001-001,A,Sample,projectB_postproc_0001.xml,870,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Synthetic INGR ISO-15926 PostProc export (project B), OriginatingSystem=SPPID.\n PipingNetworkSeg..."
2,A22-0007-003,B,Sample,ambiguous_no_originator.xml,495,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- No OriginatingSystem present -> exercises the SEGMENT_TAGNAME fallback.\n A PipingNetworkSegment ..."
3,None,None,Sample,malformed_truncated.xml,429,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Malformed export: the title block is broken before the revision can be read.\n Bronze must still ..."


## Delta history

In [9]:
from delta.tables import DeltaTable

try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

+-------+-----------------------+---------+-------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                    |
+-------+-----------------------+---------+-------------------------------------------------------+
|0      |2026-08-29 14:24:03.653|WRITE    |{mode -> ErrorIfExists, partitionBy -> ["ingest_date"]}|
+-------+-----------------------+---------+-------------------------------------------------------+



In [10]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code A

:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c30719c-7fc4-48a3-965e-169f82db03fd;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 193ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts

In [11]:
preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

,document_number,drawing_revision,project_code,source_filename,file_size_bytes,ingested_at,content_text
0,362-09-PR-PID-01470,01,A,362-09-01470_Dexpi.xml,12537819,2026-08-29 14:24:45.642813,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
1,362-92-PR-PID-02265,01,A,362-92-02265_Dexpi.xml,6669430,2026-08-29 14:24:45.642813,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
2,362-09-PR-PID-01050,01,A,362-09-01050_Dexpi.xml,11926337,2026-08-29 14:24:45.642813,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
3,362-09-PR-PID-01010,01,A,362-09-01010_Dexpi.xml,11819694,2026-08-29 14:24:45.642813,"<?xml version=""1.0"" encoding=""utf-8""?>\r\n<PlantModel xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"">\r\n <!--Created with INGR ISO15926 P..."
4,215777C-36209-PID-0021-01010,01,Sample,projectA_dexpi_01010.xml,1040,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Synthetic DEXPI/Proteus P&ID export (project A). Structure is illustrative;\n confirm real elemen..."
5,A14-0001-001,A,Sample,projectB_postproc_0001.xml,870,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Synthetic INGR ISO-15926 PostProc export (project B), OriginatingSystem=SPPID.\n PipingNetworkSeg..."
6,A22-0007-003,B,Sample,ambiguous_no_originator.xml,495,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- No OriginatingSystem present -> exercises the SEGMENT_TAGNAME fallback.\n A PipingNetworkSegment ..."
7,None,None,Sample,malformed_truncated.xml,429,2026-08-29 14:23:51.431827,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!-- Malformed export: the title block is broken before the revision can be read.\n Bronze must still ..."


In [12]:
try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                     |
+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |2026-08-29 14:24:50.893|MERGE    |{predicate -> ["(content_hash#1211 = content_hash#31)"], matchedPredicates -> [], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|0      |2026-08-29 14:24:03.653|WRITE    |{mode -> ErrorIfExists, partitionBy -> ["ingest_date"]}                                                  

In [13]:
# Optional: time-travel example for version 0 if it exists
try:
    version_zero = spark.read.format("delta").option("versionAsOf", 0).load(str(table_path))
    print("Rows in version 0:", version_zero.count())
    version_zero.show(5, truncate=False)
except Exception as e:
    print(f"Version travel is unavailable yet: {e}")
finally:
    # keep the session alive until you explicitly stop it
    pass

Rows in version 0: 4
+------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
spark.stop()